##### Partie 1 : Préparation des données

In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

#1. Charger le dataset Breast Cancer depuis scikit-learn.

breast_cancer = load_breast_cancer()
X = breast_cancer.data
y = breast_cancer.target

#2. Séparer les données en ensembles d’entraînement et de test (80% / 20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#3.Afficher la taille des ensembles obtenus
print(f"Taille des ensembles obtenus : {X_train.shape[0]} échantillons")

Taille des ensembles obtenus : 455 échantillons


##### Partie 2 : Modèle Baseline

In [2]:
#1. Créer un modèle RandomForestClassifier avec random_state=42.
rf_model = RandomForestClassifier(random_state=42)

#2. Entraîner le modèle sur les données d’entraînement.
rf_model.fit(X_train, y_train)
#. Calculer et afficher l’accuracy sur les données de test
accuracy = rf_model.score(X_test, y_test)
print(f"Accuracy du modèle sur les données de test : {accuracy:.4f}")


Accuracy du modèle sur les données de test : 0.9649


##### Partie 3 : GridSearchCV

In [3]:
#1. Définir une grille d’hyperparamètres incluant :
#- n_estimators
#- max_depth
#- min_samples_split
#- min_samples_leaf


#preparer pipeline
pipeline_rf = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestClassifier(random_state=42))
])
#creer search space
param_grid = {
    'rf': [RandomForestClassifier(random_state=42)],    
    'rf__n_estimators': [100, 200, 300],
    'rf__max_depth': [10, 12, 15],
    'rf__min_samples_split': [2, 5, 10],
    'rf__min_samples_leaf': [1, 2, 4]
}

#2. Implémenter GridSearchCV avec validation croisée (cv=5).
grid_Search = GridSearchCV(pipeline_rf, param_grid, cv=5 , verbose=1)
#Lancement de recherche
grid_Search.fit(X_train, y_train)


Fitting 5 folds for each of 81 candidates, totalling 405 fits


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('rf',
                                        RandomForestClassifier(random_state=42))]),
             param_grid={'rf': [RandomForestClassifier(random_state=42)],
                         'rf__max_depth': [10, 12, 15],
                         'rf__min_samples_leaf': [1, 2, 4],
                         'rf__min_samples_split': [2, 5, 10],
                         'rf__n_estimators': [100, 200, 300]},
             verbose=1)

In [4]:
#3. Afficher les meilleurs hyperparamètres trouvés.
print("Meilleurs hyperparamètres : ", grid_Search.best_params_)
print("Meilleur performance: ",f"{grid_Search.best_score_:.4f}")
#4. Évaluer le meilleur modèle sur le test set et afficher l’accuracy.
best_model = grid_Search.best_estimator_
test_accuracy = best_model.score(X_test, y_test)
print(f"Precision sur les données de test : {test_accuracy:.4f}")  

Meilleurs hyperparamètres :  {'rf': RandomForestClassifier(random_state=42), 'rf__max_depth': 10, 'rf__min_samples_leaf': 1, 'rf__min_samples_split': 2, 'rf__n_estimators': 200}
Meilleur performance:  0.9626
Precision sur les données de test : 0.9649


##### Partie 4 : RandomizedSearchCV

In [5]:
from sklearn.model_selection import RandomizedSearchCV
import numpy as np

# 1. Définir un espace de recherche plus large
# On utilise des listes ou des distributions pour plus de variété
param_dist = {
    'rf__n_estimators': [50, 100, 200, 400, 600],
    'rf__max_depth': [None, 5, 10, 15, 20],
    'rf__min_samples_split': [2, 5, 10],
    'rf__min_samples_leaf': [1, 2, 4]
}

# 2. Implémenter RandomizedSearchCV
random_search = RandomizedSearchCV(
    pipeline_rf, 
    param_distributions=param_dist, 
    n_iter=20, 
    cv=5, 
    random_state=42, 
    verbose=1
)

random_search.fit(X_train, y_train)

# 3. Afficher les meilleurs hyperparamètres et l'accuracy
print("Meilleurs hyperparamètres (Randomized) : ", random_search.best_params_)
print(f"Meilleure accuracy sur test set : {random_search.score(X_test, y_test):.4f}")

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Meilleurs hyperparamètres (Randomized) :  {'rf__n_estimators': 200, 'rf__min_samples_split': 2, 'rf__min_samples_leaf': 1, 'rf__max_depth': 20}
Meilleure accuracy sur test set : 0.9649


Partie 5 : Analyse et Comparaison

In [7]:
# Partie 5 : Comparaison des trois modèles

print("=" * 50)
print("Comparaison des modèles")
print("=" * 50)
print(f"Baseline accuracy        : 0.9649")
print(f"GridSearchCV accuracy    : 0.9649")
print(f"RandomizedSearchCV accuracy : 0.9649")
print()
print("Les trois modèles atteignent la même accuracy de 96.49% sur ce dataset")
print("La méthode la plus coûteuse est GridSearchCV")
print("car elle teste TOUTES les combinaisons (405 fits)")
print("contre seulement 100 fits pour RandomizedSearchCV.")

Comparaison des modèles
Baseline accuracy        : 0.9649
GridSearchCV accuracy    : 0.9649
RandomizedSearchCV accuracy : 0.9649

Les trois modèles atteignent la même accuracy de 96.49% sur ce dataset
La méthode la plus coûteuse est GridSearchCV
car elle teste TOUTES les combinaisons (405 fits)
contre seulement 100 fits pour RandomizedSearchCV.
